In [ ]:
# %pip -q install pypdf
# %pip -q install chromadb
# %pip -q install faiss-cpu
# %pip -q install pinecone

In [ ]:
from dotenv import load_dotenv
load_dotenv()

### Chroma 실습

In [ ]:
from pathlib import Path
Path.cwd()

In [ ]:
pdf_path = "../docs/knu.pdf"
Path(pdf_path).exists()

In [ ]:
from langchain_community.document_loaders import PyPDFLoader
loader = PyPDFLoader(pdf_path)
documents = loader.load()
len(documents)

In [ ]:
documents[0].metadata

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100,
)
chunks = text_splitter.split_documents(documents)
len(documents), len(chunks)

In [ ]:
chunks[0].page_content

In [ ]:
chunks[1].page_content

In [ ]:
for i, chunk in enumerate(chunks):
    chunk.metadata["chunk_id"] = i

In [ ]:
for i, chunk in enumerate(chunks[100:103]):
    print(f"[Chunk {100 + i}]")
    print(f"길이: {len(chunk.page_content)}")
    print(f"청크 ID: {chunk.metadata['chunk_id']}")
    print(chunk.page_content[:60])
    print("-" * 50)

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
)

In [ ]:
from langchain_chroma import Chroma
db_path = "chroma_db"
if Path(db_path).exists():
    vectorstore = Chroma(
        embedding_function=embeddings,
        persist_directory=db_path,
    )
else:
    vectorstore = Chroma.from_documents(
        documents=chunks,
        embedding=embeddings,
        persist_directory=db_path,
    )
vectorstore

In [ ]:
vectorstore._collection.count()

In [ ]:
list(Path(db_path).iterdir())

In [ ]:
query = "교양 과목의 이수 학점은?"
results = vectorstore.similarity_search(query, k=5)
for i, doc in enumerate(results, start=1):
    print(f"[검색 결과 {i}]")
    print(f"페이지: {doc.metadata.get('page')}")
    print(f"청크 ID: {doc.metadata.get('chunk_id')}")
    print(doc.page_content[:500])
    print("-" * 50)

확인해야 할 것:
- 질문과 관련 있는 내용이 검색되었는가?
- 검색된 청크는 어느 페이지에서 왔는가?
- 서로 다른 청크가 반환되었는가?
- 검색 결과가 질문에 답하는 데 충분한가?

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
prompt = ChatPromptTemplate.from_template(
    """
    다음 문맥만 이용하여 질문에 답하세요.
    문맥에서 답을 찾을 수 없으면
    "문서에서 답을 찾을 수 없습니다."라고 답하세요.
    [문맥] {context}
    [질문] {question}
    [답변] 
    """
)

In [ ]:
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

In [ ]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 5})
docs = retriever.invoke(query)
context = format_docs(docs)

In [ ]:
from langchain_core.output_parsers import StrOutputParser
from langchain_openai import ChatOpenAI
llm = ChatOpenAI(model="gpt-5.4-nano", temperature=0)
chain = prompt | llm | StrOutputParser()
answer = chain.invoke({
    "context": context,
    "question": query,
})
answer

In [ ]:
query = "사범대학 학생이 졸업 및 수료에 필요한 성적평점평균은?"
results = vectorstore.similarity_search(query, k=10)
for i, doc in enumerate(results, start=1):
    print(f"[검색 결과 {i}]")
    print(f"페이지: {doc.metadata.get('page')}")
    print(f"청크 ID: {doc.metadata.get('chunk_id')}")
    print(doc.page_content[:500])
    print("-" * 50)

In [ ]:
docs = retriever.invoke(query)
context = format_docs(docs)
answer = chain.invoke({
    "context": context,
    "question": query,
})
answer

### FAISS 실습

In [ ]:
from langchain_community.vectorstores import FAISS
from pathlib import Path

faiss_path = Path("faiss_db")

In [ ]:
index_file = faiss_path / "index.faiss"
pickle_file = faiss_path / "index.pkl"

index_file.exists(), pickle_file.exists()

In [ ]:
if index_file.exists() and pickle_file.exists():
    vectorstore = FAISS.load_local(
        folder_path=str(faiss_path),
        embeddings=embeddings,
        allow_dangerous_deserialization=True,
    )
    print("기존 FAISS Vector Store를 불러왔습니다.")
else:
    vectorstore = FAISS.from_documents(
        documents=chunks,
        embedding=embeddings,
    )

    vectorstore.save_local(str(faiss_path))
    print("새로운 FAISS Vector Store를 생성했습니다.")

In [ ]:
print(vectorstore.index.ntotal)
print(type(vectorstore.index))
print(type(vectorstore.docstore))

In [ ]:
list(vectorstore.index_to_docstore_id.items())[:3]

In [ ]:
scored_results = vectorstore.similarity_search_with_score(query, k=5)
for i, (doc, score) in enumerate(scored_results, start=1):
    page = doc.metadata.get("page")
    page_number = page + 1 if page is not None else "?"
    print(f"[검색 결과 {i}]")
    print(f"거리: {score:.4f}")
    print(f"PDF 페이지: {page_number}")
    print(doc.page_content[:300])
    print("-" * 50)

In [ ]:
query = "교양 과목의 이수 학점은?"
retriever = vectorstore.as_retriever(search_kwargs={"k": 5})
docs = retriever.invoke(query)
context = format_docs(docs)
chain = prompt | llm | StrOutputParser()
answer = chain.invoke({
    "context": context,
    "question": query,
})
answer

In [ ]:
query = "사범대학 학생이 졸업 및 수료에 필요한 성적평점평균은?"
docs = retriever.invoke(query)
context = format_docs(docs)
chain = prompt | llm | StrOutputParser()
answer = chain.invoke({
    "context": context,
    "question": query,
})
answer

### Pinecone 실습

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()
key = os.getenv("PINECONE_API_KEY")
key[:10]

In [ ]:
from datasets import load_dataset

dp_dataset = load_dataset('klue/klue', 'dp', split='train[:100]')
for key, value in dp_dataset[0].items():
    print(f"{key}: {value[:5]}")

In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('snunlp/KR-SBERT-V40K-klueNLI-augSTS')
embeddings = model.encode(dp_dataset['sentence'])

In [ ]:
from pinecone import Pinecone, ServerlessSpec

pc = Pinecone()

# Pinecone 웹사이트에서 생성
if not pc.has_index("klue-dp-demo"):
    pc.create_index(
        name="klue-dp-demo",
        spec=ServerlessSpec(
            cloud="aws",
            region="us-east-1"),
        dimension=768,
        metric="cosine"
    )

In [ ]:
sentences = [row["sentence"] for row in dp_dataset]
vectors = [
    (str(i), embeddings[i].tolist(), {'text': sentences[i]})
    for i in range(len(sentences))
]

In [ ]:
index = pc.Index('klue-dp-demo')
index.upsert(
    vectors=vectors,
    namespace='klue-dp-demo-sub'
)
# Pinecone 웹사이트에서 입력 결과 확인

In [ ]:
response = index.query(
    namespace="klue-dp-demo-sub",
    top_k=10,
    include_values=True,
    include_metadata=True,
    vector=embeddings[0].tolist()
)

In [ ]:
for i, match in enumerate(response["matches"], start=1):
    print(f"{i}. score={match['score']:.4f}")
    print(match["metadata"]["text"])